In [2]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import json

def age_to_group(age):

    if age < 18:
        return "0-18"
    elif age <= 40:
        return "18-40"
    elif age <= 60:
        return "40-60"
    elif age <= 80:
        return "60-80"
    else:
        return "80+"

def stratify_group(row):
    # Combine demographic and label attributes into a single stratification key
    attrs = [
        str(row["gender"]).strip().upper() if pd.notna(row["gender"]) else "UNK",
        str(row["ethnicity"]).strip().upper() if pd.notna(row["ethnicity"]) else "UNK",
        str(row['age_group']).strip().upper() if pd.notna(row["age_group"]) else "UNK",
    ]

    return "_".join(attrs)

source_dir = Path("/home/vito/ibrahimm/projects/AI4Health/sourcedata/Chest Xray/physionet.org/files/mimic-cxr-jpg/2.0.0/")
raw_dir = Path("/home/vito/ibrahimm/projects/AI4Health/notebooks/ibrahimm/Generative-Models/images/Chest_XRay/my_work/raw_data")
resize_size = 512

original_metadata_df = pd.read_csv(raw_dir / "mimic-cxr-2.0.0-metadata-with-demographics.csv")
original_metadata_df['age_group'] = original_metadata_df['anchor_age'].apply(age_to_group)

chexpert_df = pd.read_csv(raw_dir / "mimic-cxr-2.0.0-chexpert.csv")
impressions_df = pd.read_csv("/home/vito/ibrahimm/projects/AI4Health/notebooks/ibrahimm/Generative-Models/images/Chest_XRay/RoentGen-v2/real_data/sections/mimic_cxr_sectioned.csv")
impressions_df['study_id'] = impressions_df['study'].str.replace('s', '').astype(int)
merged_df = pd.merge(original_metadata_df, chexpert_df, on=["subject_id", "study_id"], how="inner")
# roentgen-v2 use impressions
all_df = pd.merge(impressions_df, merged_df, on="study_id", how="inner")
all_df['image'] = all_df.apply(
    lambda row: f"{raw_dir}/files/p{str(int(row['subject_id']))[:2]}/p{str(int(row['subject_id']))}/"
    f"s{int(row['study_id'])}/{row['dicom_id']}.jpg", 
    axis=1)
all_df['folder'] = all_df['subject_id'].apply(lambda x: str(x)[:2])
all_df.shape


(377024, 37)

In [3]:
# --- Filter and prepare PA_data as before ---
PA_data = all_df[all_df['ViewPosition'] == 'PA']
print('Data after excluding non-PA studies:', PA_data.shape)
PA_data = PA_data[PA_data['impression'].notna()]
print('Data after excluding studies with no impression:', PA_data.shape)
ethnicity_map = {
    'WHITE': 'White',
    'HISPANIC/LATINO': 'Hispanic', 
    'BLACK/AFRICAN AMERICAN': 'Black',
    'ASIAN': 'Asian'
}
PA_data['ethnicity'] = PA_data['ethnicity'].map(ethnicity_map)
ethnicity_to_drop = ['UNKNOWN', 'OTHER', 'UNABLE TO OBTAIN','AMERICAN INDIAN/ALASKA NATIVE']
PA_data = PA_data.dropna(subset=['ethnicity'])
PA_data['impression_length'] = PA_data['impression'].str.len()
PA_data = PA_data[~PA_data['ethnicity'].isin(ethnicity_to_drop)]
print('Data after excluding specified ethnicity values:', PA_data.shape)
PA_data['gender'] = PA_data['gender'].map({'M': 'male', 'F': 'female'})
PA_data['sentence'] = PA_data.apply(lambda row: f"{int(row['anchor_age'])} year old {row['ethnicity']} {row['gender']}. {row['impression']}", axis=1)


# Create stratification key
PA_data["stratify_key"] = PA_data.apply(stratify_group, axis=1)



Data after excluding non-PA studies: (96143, 37)
Data after excluding studies with no impression: (87056, 37)
Data after excluding specified ethnicity values: (70433, 38)


In [5]:
PA_data['gender'].unique()

array(['female', 'male'], dtype=object)

In [6]:
PA_data['ethnicity'].unique()

array(['White', 'Black', 'Asian', 'Hispanic'], dtype=object)

In [7]:
PA_data['impression'].unique()

array(['No acute cardiopulmonary abnormality.',
       'No acute cardiopulmonary process.', 'Stable chest radiograph.',
       ...,
       'Persistent small right pleural effusion.  Otherwise unremarkable. Pacemaker in\n unchanged position.',
       'Persistent small bilateral effusions.',
       'PA and lateral chest compared to a series of chest CT scans, most\n recently ___:\n \n ___ x 22 mm elliptical opacity projecting over the cardiac apex on the frontal\n view could be the lung nodule referenced.  I cannot identify it on the\n lateral.  Loss of volume in the right lung is attributable to right upper\n lobectomy and obscuration of the right heart border and anterior aspect of the\n right hemidiaphragm could be due to anatomic rearrangement stemming from\n surgery.  Upper lungs are clear.  The heart is normal size.  Thoracic aorta is\n mildly dilated and very tortuous.  There is no pleural effusion.  I see no\n explanation for decreased breath sounds in the left lower chest, excep

# Tokenizer

In [8]:
# Tokenize impressions in PA_data using Stable Diffusion's tokenizer
from transformers import AutoTokenizer
token = 'hf_TOKEN_REMOVED'
# Choose the tokenizer consistent with training code (CLIP tokenizer from SD v1-4)
model_id = "stanfordmimi/RoentGen-v2"
try:
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        subfolder="tokenizer",
        use_fast=True,
        trust_remote_code=True,
        token=token,
    )
except Exception:
    # Fallback to default loading if subfolder is not available
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True, token=token)

# Ensure PA_data exists and has an 'impression' column
assert 'PA_data' in globals(), "PA_data DataFrame not found in the notebook."
assert 'impression' in PA_data.columns, "PA_data is missing the 'impression' column."

# Prepare texts (handle NaNs)
texts = PA_data['impression'].fillna("").astype(str)

# Get tokenizer max length (defaults to 77 for CLIP)
if hasattr(tokenizer, 'model_max_length'):
    max_len = tokenizer.model_max_length
else:
    max_len = 77  # fallback CLIP default

# Batch tokenize for efficiency (with truncation and max_length)
encodings = tokenizer(
    texts.tolist(),
    padding='max_length',
    truncation=True,
    max_length=max_len,
    return_attention_mask=True,
)

# Store tokenized outputs as list columns
PA_data['input_ids'] = encodings['input_ids']
PA_data['attention_mask'] = encodings['attention_mask']

# Check for entries that exceeded the token limit BEFORE truncation (by re-tokenizing with no truncation)
lengths = [len(tokenizer.encode(t, add_special_tokens=True, truncation=False)) for t in texts]
PA_data['n_input_tokens'] = lengths
PA_data['truncated'] = PA_data['n_input_tokens'] > max_len

# Show stats about truncation
n_truncated = PA_data['truncated'].sum()
print(f"Number of samples truncated due to token limit ({max_len}): {n_truncated}")
if n_truncated > 0:
    print(PA_data.loc[PA_data['truncated'], ['impression', 'n_input_tokens']].head())

# Quick sanity check: show shapes/lengths of the first few entries
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Max length used:", max_len)
print("First input_ids length:", len(PA_data['input_ids'].iloc[0]) if len(PA_data) else None)


Token indices sequence length is longer than the specified maximum sequence length for this model (80 > 77). Running this sequence through the model will result in indexing errors


Number of samples truncated due to token limit (77): 3939
                                            impression  n_input_tokens
25   Compared to chest radiographs since ___, most ...              80
335  PA and lateral chest reviewed in the absence o...             123
574  Compared to chest radiographs ___ and chest CT...              91
709  Pacemaker leads terminate in right atrium and ...              89
744  PA and lateral chest compared to ___.\n \n Sli...              79
Tokenizer vocab size: 49408
Max length used: 77
First input_ids length: 77


In [10]:
PA_data[PA_data['truncated']==True]['impression'].to_csv('long_impressions.csv')

In [9]:
long_impressions = PA_data[PA_data['truncated']==True]
long_impressions = long_impressions.reset_index(drop=False)
long_impressions = long_impressions.reset_index(drop=False)

In [11]:


import json

# Read the output_impressions_openai.jsonl file, where each line is a JSON object as in lines 1-2
input_to_openai = []
with open("long_impressions_openai_chat_batch.jsonl", "r", encoding="utf-8") as fin:
    for line in fin:
        # skip blank lines
        if line.strip():
            input_to_openai.append(json.loads(line))
input_to_openai = pd.DataFrame(input_to_openai)
mapping = pd.concat([input_to_openai, long_impressions], axis=1)[['custom_id', 'index']]

In [12]:
payload_input_path = "long_impressions_openai_chat_batch.jsonl"  # This is the same as used for OpenAI input

# Load the mapping from input payload (e.g., includes which study/impression each line is for)
payloads = []
with open(payload_input_path, "r") as f:
    for line in f:
        payloads.append(json.loads(line))


payload_df = pd.DataFrame(payloads)

impressions_with_id = PA_data[PA_data['truncated']==True]['impression'].reset_index().rename(columns={"index": "impression_id"})

concatenated = pd.concat([payload_df, impressions_with_id], axis=1)
concatenated

,custom_id,method,url,body,impression_id,impression
0,request-1,POST,/v1/chat/completions,"{'model': 'gpt-5', 'messages': [{'role': 'syst...",25,"Compared to chest radiographs since ___, most ..."
1,request-2,POST,/v1/chat/completions,"{'model': 'gpt-5', 'messages': [{'role': 'syst...",335,PA and lateral chest reviewed in the absence o...
2,request-3,POST,/v1/chat/completions,"{'model': 'gpt-5', 'messages': [{'role': 'syst...",574,Compared to chest radiographs ___ and chest CT...
3,request-4,POST,/v1/chat/completions,"{'model': 'gpt-5', 'messages': [{'role': 'syst...",709,Pacemaker leads terminate in right atrium and ...
4,request-5,POST,/v1/chat/completions,"{'model': 'gpt-5', 'messages': [{'role': 'syst...",744,PA and lateral chest compared to ___.\n \n Sli...
...,...,...,...,...,...,...
3934,request-3935,POST,/v1/chat/completions,"{'model': 'gpt-5', 'messages': [{'role': 'syst...",376894,"In comparison with the study of ___, the right..."
3935,request-3936,POST,/v1/chat/completions,"{'model': 'gpt-5', 'messages': [{'role': 'syst...",376898,PA and lateral chest compared to ___:\n \n Sma...
3936,request-3937,POST,/v1/chat/completions,"{'model': 'gpt-5', 'messages': [{'role': 'syst...",376912,"As compared to the previous radiograph, there ..."
3937,request-3938,POST,/v1/chat/completions,"{'model': 'gpt-5', 'messages': [{'role': 'syst...",376920,Status post exchange of a right pectoral Port-...


In [37]:
import json

# Read the output_impressions_openai.jsonl file, each line is a JSON object as in lines 1-2
raw_rows = []
contents = []
with open("output_impressions_openai_batch1.jsonl", "r", encoding="utf-8") as fin:
    for line in fin:
        if line.strip():
            data = json.loads(line)
            # Defensive: check nested structure per @file_context_0
            try:
                content = data['response']['body']['choices'][0]['message']['content']
                content_stripped = content.strip() if content else content
                # Add stripped content to a new column, leave raw JSON as is
                contents.append(content_stripped)
            except KeyError:
                contents.append(None)
            raw_rows.append(data)

df = pd.DataFrame(raw_rows)
df['summarized'] = contents
successful_summarized = df.merge(mapping, left_on='custom_id', right_on='custom_id', how='left')[['index', 'summarized']]
successful_summarized

,index,summarized
0,25,Compared to prior CXRs: edema and possible pne...
1,1497,Heart/mediastinum stable. Bilateral multifocal...
2,2867,"Decreased bilateral pleural effusions, now sma..."
3,2881,Right lower mediastinal contour enlarged; like...
4,2998,Cardiac size stable. Basal interstitial markin...
...,...,...
654,368132,R subclavian catheter distal SVC. Mild cardiom...
655,371016,"Low lung volumes. Patchy opac L mid/lower, R b..."
656,375490,"Persistent moderate multiloculated L effusion,..."
657,375504,Large left pleural effusion with new right med...


In [38]:
merged_PA_data = PA_data.reset_index(drop=False).merge(successful_summarized, left_on='index', right_on='index', how='left')
merged_PA_data.to_csv('summarized_PA_data.csv', index=False)


all data with partially (659) summarized data written to summarized_PA_data.csv

In [40]:
need_to_be_summarized = merged_PA_data[(merged_PA_data['truncated'] == True) & (merged_PA_data['summarized'].isna())]

In [41]:
impressions_df = need_to_be_summarized.reset_index()

In [42]:
original_to_remain_map = impressions_df[['level_0', 'index']]
original_to_remain_map.rename(columns={'level_0': 'index_in_remaining', 'index': 'index_in_original'}, inplace=True)

/tmp/ipykernel_1144876/30911549.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  original_to_remain_map.rename(columns={'level_0': 'index_in_remaining', 'index': 'index_in_original'}, inplace=True)


# merge

In [44]:
summarized_PA_data = pd.read_csv('summarized_PA_data.csv')
summarized_PA_data.shape

(70433, 46)

In [45]:
mapping_df = pd.read_csv('long_impressionid_customid_mapping.csv')
mapping_df = mapping_df.merge(original_to_remain_map, left_on='impression_id', right_on='index_in_remaining', how='left')
mapping_df.head()

,impression_id,custom_id,index_in_remaining,index_in_original
0,59,request-1,59,335
1,108,request-2,108,574
2,126,request-3,126,709
3,135,request-4,135,744
4,154,request-5,154,894


In [46]:
summarized_batch2_with_keys = summarized_batch2.merge(mapping_df, left_on='custom_id', right_on='custom_id', how='left')
summarized_batch2_with_keys

,id,custom_id,summarized,impression_id,index_in_remaining,index_in_original
0,batch_req_69167e88940c81908091efbeba3aff8a,request-1,CXR: Moderate bilateral pleural effusions and ...,59,59,335
1,batch_req_69167e88ddb881909cd4e975acc8d2dd,request-2,"Vs prior CXR/CT: Lungs clear, fully expanded. ...",108,108,574
2,batch_req_69167e888f988190869592016c1b9b11,request-3,Pacemaker leads in RA/RV. Stable heart/mediast...,126,126,709
3,batch_req_69167e889f8c8190a47f536dd442e9a1,request-4,Increased pulmonary vascular congestion sugges...,135,135,744
4,batch_req_69167e88892481908221e49ed6d05f6d,request-5,Left pigtail catheter no longer seen; no pneum...,154,154,894
...,...,...,...,...,...,...
3275,batch_req_6916818095a08190bfb2a023f3d219f5,request-3276,Right base now clear except atelectasis/fibros...,70410,70410,376894
3276,batch_req_691681810a648190b21fe02c3d0d1853,request-3277,Small-moderate bilateral pleural effusions (R>...,70411,70411,376898
3277,batch_req_69168180dec48190ba397c2cd4e7b336,request-3278,Vs prior: slight increase R pleural effusion a...,70412,70412,376912
3278,batch_req_69168180c7bc8190b6a1854905962490,request-3279,Right chest port exchanged; tip at right atriu...,70414,70414,376920


In [47]:
all_summarized_PA_data = summarized_PA_data.merge(summarized_batch2_with_keys, left_on='index', right_on='index_in_original', how='left')


In [48]:
all_summarized_PA_data[all_summarized_PA_data['truncated'] == True].shape

(3939, 52)

In [49]:
print(all_summarized_PA_data[~all_summarized_PA_data['summarized_x'].isna()].shape)
print(all_summarized_PA_data[~all_summarized_PA_data['summarized_y'].isna()].shape)

(659, 52)
(3280, 52)


In [51]:
import numpy as np

all_summarized_PA_data['summarized'] = all_summarized_PA_data['summarized_x'].combine_first(all_summarized_PA_data['summarized_y'])


In [52]:
all_summarized_PA_data.to_csv('raw_all_summarized_PA_data.csv', index=False)

In [53]:
all_summarized_PA_data.columns

Index(['index', 'study', 'impression', 'findings', 'last_paragraph',
       'comparison', 'study_id', 'dicom_id', 'subject_id',
       'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns',
       'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning',
       'ViewCodeSequence_CodeMeaning',
       'PatientOrientationCodeSequence_CodeMeaning', 'gender', 'anchor_age',
       'ethnicity', 'age_group', 'Atelectasis', 'Cardiomegaly',
       'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture',
       'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion',
       'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices',
       'image', 'folder', 'impression_length', 'sentence', 'stratify_key',
       'input_ids', 'attention_mask', 'n_input_tokens', 'truncated',
       'summarized_x', 'id', 'custom_id', 'summarized_y', 'impression_id',
       'index_in_remaining', 'index_in_original', 'summarized'],
      dtype='object')

In [54]:
clean = all_summarized_PA_data[['index', 'study', 'impression', 'findings', 'last_paragraph',
       'comparison', 'study_id', 'dicom_id', 'subject_id',
       'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns',
       'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning',
       'ViewCodeSequence_CodeMeaning',
       'PatientOrientationCodeSequence_CodeMeaning', 'gender', 'anchor_age',
       'ethnicity', 'age_group', 'Atelectasis', 'Cardiomegaly',
       'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture',
       'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion',
       'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices',
       'image', 'folder', 'impression_length', 'sentence', 'stratify_key',
       'input_ids', 'attention_mask', 'n_input_tokens', 'truncated',
        'summarized']]
clean.to_csv('clean_all_summarized_PA_data.csv', index=False)

In [63]:
clean[clean['truncated'] == True][['study', 'subject_id', 'impression', 'summarized']].to_csv('summarized_PA_data.csv', index=False)

In [173]:
all_summarized_PA_data[(all_summarized_PA_data['truncated'] == True) & (all_summarized_PA_data['summarized'].isna())]

KeyError: 'summarized'

# openai

## A. prepare

In [71]:
import json

# Prepare a batch jsonl file for OpenAI /v1/chat/completions endpoint and create mapping between impression id and custom id
batch_jsonl_filename = "long_impressions_openai_chat_batch_2.jsonl"
impression_customid_mapping_filename = "long_impressionid_customid_mapping.csv"
system_message = "You are a helpful assistant."
user_task = 'Your task is to summarize this radiology report within 200 characters or less. Your response must be concise, truthful, and keep all relevant medical information.'
model_name = "gpt-5"  # Or set to your desired model

impressions_df = need_to_be_summarized['impression'].reset_index()  # include index (impression id)
custom_id_list = []
impression_id_list = []

with open(batch_jsonl_filename, "w", encoding="utf-8") as fout:
    for idx, row in impressions_df.iterrows():
        imp = row['impression']
        impression_id = row['index']
        custom_id = f"request-{idx + 1}"
        payload = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": f"{user_task}\n\n{imp}"}
                ]
            }
        }
        fout.write(json.dumps(payload) + "\n")
        # Collect mapping for this request
        impression_id_list.append(impression_id)
        custom_id_list.append(custom_id)

# Save the mapping to CSV for later correspondence
import pandas as pd
mapping_df = pd.DataFrame({"impression_id": impression_id_list, "custom_id": custom_id_list})
mapping_df.to_csv(impression_customid_mapping_filename, index=False)

print(f"Wrote {len(impressions_df)} requests to {batch_jsonl_filename} for OpenAI batch /v1/chat/completions.")
print(f"Mapping between impression_id and custom_id written to {impression_customid_mapping_filename}.")


Wrote 3280 requests to long_impressions_openai_chat_batch_2.jsonl for OpenAI batch /v1/chat/completions.
Mapping between impression_id and custom_id written to long_impressionid_customid_mapping.csv.


## B. Submit

In [87]:
from openai import OpenAI
client = OpenAI(
  api_key="sk-TOKEN_REMOVED"
)

batch_input_file = client.files.create(
    file=open("long_impressions_openai_chat_batch_2.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-H8USqtNQM5ThWxqHZzfvCz', bytes=2857720, created_at=1763045430, filename='long_impressions_openai_chat_batch_2.jsonl', object='file', purpose='batch', status='processed', expires_at=1765637430, status_details=None)


In [91]:
from openai import OpenAI

batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "summarize long impressions"
    }
)

Batch(id='batch_6915f0f75f108190b3b47c306572a2de', completion_window='24h', created_at=1763045623, endpoint='/v1/chat/completions', input_file_id='file-H8USqtNQM5ThWxqHZzfvCz', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1763132023, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'summarize long impressions'}, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))

In [2]:
import time
from openai import OpenAI
client = OpenAI(
  api_key="sk-TOKEN_REMOVED"
)

batch_id = "batch_6915f0f75f108190b3b47c306572a2de"

while True:
    batch = client.batches.retrieve(batch_id)
    print(f"Batch: {batch}")
    print(f"Status: {batch.status}")
    print(f"Batch counts: {batch.request_counts}")
    if batch.status in ["completed", "failed", "cancelled"]:
        print("Batch processing finished.")
        break
    time.sleep(60)  # wait 1 minute before checking again

Batch: Batch(id='batch_6915f0f75f108190b3b47c306572a2de', completion_window='24h', created_at=1763045623, endpoint='/v1/chat/completions', input_file_id='file-H8USqtNQM5ThWxqHZzfvCz', object='batch', status='in_progress', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1763132023, failed_at=None, finalizing_at=None, in_progress_at=1763045630, metadata={'description': 'summarize long impressions'}, model='gpt-5-2025-08-07', output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=3280), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))
Status: in_progress
Batch counts: BatchRequestCounts(completed=0, failed=0, total=3280)
Batch: Batch(id='batch_6915f0f75f108190b3b47c306572a2de', completion_window='24h', created_at=1763045623, endpoint='/v1/chat/completion

In [7]:
client.batches.retrieve(batch_id)


Batch(id='batch_6915f0f75f108190b3b47c306572a2de', completion_window='24h', created_at=1763045623, endpoint='/v1/chat/completions', input_file_id='file-H8USqtNQM5ThWxqHZzfvCz', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1763082626, error_file_id=None, errors=None, expired_at=None, expires_at=1763132023, failed_at=None, finalizing_at=1763081860, in_progress_at=1763045630, metadata={'description': 'summarize long impressions'}, model='gpt-5-2025-08-07', output_file_id='file-EzC6NZv5SfcvckqVyZggj5', request_counts=BatchRequestCounts(completed=3280, failed=0, total=3280), usage=BatchUsage(input_tokens=504855, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=10728350, output_tokens_details=OutputTokensDetails(reasoning_tokens=10546560), total_tokens=11233205))

## C. Retreive

In [30]:
import json
import pandas as pd
from openai import OpenAI
client = OpenAI(
  api_key="sk-TOKEN_REMOVED"
)
import json

file_response = client.files.content("file-EzC6NZv5SfcvckqVyZggj5")
output_path = "output_impressions_openai_batch2.jsonl"

with open(output_path, "w") as f:
    for line in file_response.text.strip().splitlines():
        # Each line should already be json-serialized; optionally validate/parsing:
        try:
            obj = json.loads(line)
            f.write(json.dumps(obj) + "\n")
        except json.JSONDecodeError:
            # If it's plain text, wrap in JSON
            f.write(json.dumps({"text": line}) + "\n")

print(f"Wrote file to {output_path}")


# Extract relevant fields from jsonl lines and build list of dicts
records = []
for line in file_response.text.strip().splitlines():
    try:
        obj = json.loads(line)
        # get id and summary content
        record = {
            "id": obj.get("id"),
            "custom_id": obj.get("custom_id"),
        }
        # Try to extract summary text (OpenAI batch response format)
        try:
            record["summarized"] = obj["response"]["body"]["choices"][0]["message"]["content"]
        except Exception:
            record["summarized"] = None
        records.append(record)
    except Exception:
        pass

summarized_batch2 = pd.DataFrame(records)


Wrote file to output_impressions_openai_batch2.jsonl


In [70]:

import json

error_response = client.files.content("file-7Tsv1kJD4WfbKBnkikxH7L")
output_path = "error_impressions_openai.jsonl"

with open(output_path, "w") as f:
    for line in error_response.text.strip().splitlines():
        # Each line should already be json-serialized; optionally validate/parsing:
        try:
            obj = json.loads(line)
            f.write(json.dumps(obj) + "\n")
        except json.JSONDecodeError:
            # If it's plain text, wrap in JSON
            f.write(json.dumps({"text": line}) + "\n")

print(f"Wrote file to {output_path}")


Wrote file to error_impressions_openai.jsonl


In [69]:
batch.error_file_id


'file-7Tsv1kJD4WfbKBnkikxH7L'

In [62]:
client.batches.list()

SyncCursorPage[Batch](data=[Batch(id='batch_6911fa16eb008190aba6d16de9d3b598', completion_window='24h', created_at=1762785814, endpoint='/v1/chat/completions', input_file_id='file-GEpXe3D3cr8pDn9wGTnmno', object='batch', status='in_progress', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1762872214, failed_at=None, finalizing_at=None, in_progress_at=1762785878, metadata={'description': 'summarize long impressions'}, model='gpt-5-2025-08-07', output_file_id=None, request_counts=BatchRequestCounts(completed=663, failed=3105, total=3939), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0)), Batch(id='batch_6911f850c2b08190adf0a756a420fd4e', completion_window='24h', created_at=1762785360, endpoint='/v1/chat/completions', input_file_id='file-6wbHup5nobWMo4hooCYjBr', object='batch', sta

In [ ]:


file_response = client.files.content("file-xyz123")
print(file_response.text)